# 循环神经网络

在前一模块中，我们介绍了丰富的文本语义表示。我们一直使用的架构捕获了句子中词语的聚合意义，但它没有考虑词语的<strong>顺序</strong>，因为紧跟在嵌入之后的聚合操作会从原始文本中移除这些信息。由于这些模型无法表示词序，因此它们无法解决更复杂或含糊的任务，如文本生成或问答。

为了捕获文本序列的含义，我们将使用一种称为<strong>循环神经网络</strong>（RNN）的神经网络架构。使用 RNN 时，我们将句子一次传入网络一个标记，网络会产生某个<strong>状态</strong>，然后将该状态与下一个标记一起再次传入网络。

![显示循环神经网络生成示例的图像。](../../../../../translated_images/zh-CN/rnn.27f5c29c53d727b5.webp)

给定输入标记序列 $X_0,\dots,X_n$，RNN 创建一个神经网络模块序列，并通过反向传播端到端训练此序列。每个网络模块接受一对 $(X_i,S_i)$ 作为输入，并输出 $S_{i+1}$ 作为结果。最终状态 $S_n$ 或输出 $Y_n$ 会进入线性分类器以产生结果。所有网络模块共享同一组权重，并通过一次反向传播端到端训练。

> 上图显示了展开形式的循环神经网络（左侧）和更紧凑的循环表示（右侧）。重要的是要明白，所有 RNN 单元具有相同的<strong>可共享权重</strong>。

由于状态向量 $S_0,\dots,S_n$ 被传递通过网络，RNN 能够学习词语之间的顺序依赖关系。例如，当序列中某处出现单词 *not*，它可以学会对状态向量中的某些要素进行否定。

每个 RNN 单元内部包含两个权重矩阵：$W_H$ 和 $W_I$，以及偏置 $b$。在每个 RNN 步骤中，给定输入 $X_i$ 和输入状态 $S_i$，输出状态计算公式为 $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$，其中 $f$ 是激活函数（通常是 $\tanh$）。

> 对于诸如文本生成（我们将在下一单元讲）或机器翻译等问题，我们还希望在每个 RNN 步骤得到输出值。这时会有另一个矩阵 $W_O$，输出计算公式为 $Y_i=f(W_O\times S_i+b_O)$。

让我们看看循环神经网络如何帮助我们对新闻数据集进行分类。

> 对于沙盒环境，我们需要运行以下代码单元以确保所需库已安装且数据已预取。如果您在本地运行，可以跳过以下单元。


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

在训练大型模型时，GPU内存分配可能会成为一个问题。我们可能还需要尝试不同的小批量尺寸，以确保数据适合我们的GPU内存，同时训练速度足够快。如果您在自己的GPU机器上运行此代码，可以尝试调整小批量大小以加快训练速度。

> <strong>注意</strong>：某些版本的NVidia驱动程序已知在训练模型后不会释放内存。我们在这个笔记本中运行了几个示例，这可能会导致在某些配置中内存耗尽，尤其是当您在同一个笔记本中进行自己的实验时。如果在开始训练模型时遇到一些奇怪的错误，您可能需要重新启动笔记本内核。


In [3]:
batch_size = 16
embed_size = 64

## 简单 RNN 分类器

对于简单的 RNN，每个循环单元是一个简单的线性网络，它接收输入向量和状态向量，并产生一个新的状态向量。在 Keras 中，这可以用 `SimpleRNN` 层表示。

虽然我们可以直接将独热编码的标记传入 RNN 层，但由于其维度较高，这并不是一个好主意。因此，我们将使用嵌入层来降低词向量的维度，随后是一个 RNN 层，最后是一个 `Dense` 分类器。

> <strong>注意</strong>：在维度不是很高的情况下，例如使用字符级标记化时，直接将独热编码的标记传入 RNN 单元可能更合适。


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **注意：** 这里我们使用未训练的嵌入层以简化操作，但为了获得更好的效果，我们可以使用前面单元中描述的基于 Word2Vec 的预训练嵌入层。将此代码改编为使用预训练嵌入会是一个很好的练习。

现在让我们训练 RNN。RNN 通常较难训练，因为一旦将 RNN 单元展开到序列长度，反向传播涉及的层数就会非常多。因此我们需要选择较小的学习率，并使用更大的数据集来训练网络以获得良好结果。这可能会花费相当长的时间，因此建议使用 GPU。

为了加快速度，我们将仅在新闻标题上训练 RNN 模型，省略描述。你可以尝试用描述进行训练，看看能否让模型成功训练。


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> <strong>注意</strong> 这里的准确性可能较低，因为我们只是在新闻标题上进行训练。


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


既然我们现在使用掩码，就可以在整个标题和描述数据集上训练模型了。

> <strong>注意</strong>：你是否注意到我们一直使用的是在新闻标题上训练的向量化器，而不是整篇文章的内容？这可能会导致一些词元被忽略，因此最好重新训练向量化器。不过，这可能只有很小的影响，所以为了简单起见，我们将继续使用之前预训练的向量化器。


## LSTM：长短期记忆

RNN 的一个主要问题是<strong>梯度消失</strong>。RNN 可能非常长，且在反向传播时可能难以将梯度一直传回网络的第一层。当发生这种情况时，网络无法学习远距离标记之间的关系。避免这个问题的一种方法是通过使用<strong>门控</strong>引入<strong>显式状态管理</strong>。引入门控的两种最常见架构是<strong>长短期记忆</strong>（LSTM）和<strong>门控循环单元</strong>（GRU）。我们这里将介绍 LSTM。

![展示一个长短期记忆单元示例的图像](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

LSTM 网络的组织方式类似于 RNN，但有两个状态从层传递到层：实际状态 $c$ 和隐藏向量 $h$。在每个单元处，隐藏向量 $h_{t-1}$ 和输入 $x_t$ 结合在一起，通过<strong>门控</strong>控制状态 $c_t$ 和输出 $h_{t}$ 的变化。每个门都有 sigmoid 激活（输出范围为 $[0,1]$），可以被看作是状态向量的按位掩码。LSTM 具有以下门（如上图从左到右）：
* <strong>遗忘门</strong>，决定我们需要遗忘状态向量 $c_{t-1}$ 中的哪些成分，哪些可以传递下去。
* <strong>输入门</strong>，决定有多少来自输入向量和之前隐藏向量的信息被整合进状态向量。
* <strong>输出门</strong>，拿到新的状态向量，决定它的哪些成分将用于生成新的隐藏向量 $h_t$。

状态 $c$ 的成分可以被看作可以开关的标志。例如，当我们在序列中遇到名字 *Alice*，我们推测它指的是女性，并在状态中打开表示句子中含有女性名词的标志。当我们接着遇到 *and Tom* 这几个词，我们将打开表示含有复数名词的标志。通过操控状态，我们能够跟踪句子的语法属性。

> <strong>注意</strong>：这里有一个非常好的资源帮助理解 LSTM 内部结构：[Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/)，作者是 Christopher Olah。

虽然 LSTM 单元的内部结构看起来复杂，但 Keras 将其实现隐藏在 `LSTM` 层中，因此在上面的示例中，我们唯一需要做的就是替换循环层：


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> <strong>注意</strong> 训练 LSTM 也相当缓慢，并且您在训练初期可能不会看到准确率有太大提升。您可能需要继续训练一段时间才能达到较好的准确率。


## 双向和多层 RNN

在我们迄今为止的示例中，递归网络都是从序列的开始运行到结束。这对我们来说感觉很自然，因为它遵循了我们阅读或听语音的方向。然而，对于需要随机访问输入序列的场景，更合理的做法是让递归计算在两个方向上运行。允许双向计算的 RNN 被称为<strong>双向</strong> RNN，可以通过用特殊的 `Bidirectonal` 层封装递归层来创建。

> <strong>注意</strong>：`Bidirectional` 层在其中复制两份该层，并将其中一份的 `go_backwards` 属性设置为 `True`，使其沿序列反向进行。

递归网络，无论是单向还是双向，都能捕获序列中的模式，并将其存储为状态向量或作为输出返回。与卷积网络类似，我们可以在第一层之后构建另一层递归层，以捕获更高级的模式，而这些高级模式是由第一层提取的低级模式构成的。这引出了<strong>多层 RNN</strong>的概念，它由两个或更多递归网络组成，其中前一层的输出作为输入传递给下一层。

![展示多层长短期记忆 RNN 的图片](../../../../../translated_images/zh-CN/multi-layer-lstm.dd975e29bb2a59fe.webp)

*图片来源于 Fernando López 的[这篇精彩文章](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3)。*

Keras 使构建这些网络变得简单，因为你只需要向模型中添加更多的递归层。对于除最后一层之外的所有层，我们需要指定 `return_sequences=True` 参数，因为我们希望该层返回所有中间状态，而不仅仅是递归计算的最终状态。

让我们为我们的分类问题构建一个两层双向 LSTM。

> <strong>注意</strong> 这段代码运行时间依旧相当长，但它给出了迄今为止我们看到的最高准确率。所以也许值得等待一下，看看结果。


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNN 用于其他任务

到目前为止，我们一直专注于使用 RNN 来对文本序列进行分类。但它们可以处理更多任务，比如文本生成和机器翻译——我们将在下一单元中考虑这些任务。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
